# S6E9 | Validation Stability and ID-Order Stress Tests

**A practical check before trusting a validation score:** keep a simple model
fixed, change the split, and inspect the groups hidden behind the overall AUC.

### Results at a glance

| Verified Kaggle run, 2026-09-07 | Result |
|---|---|
| One fixed logistic model, eight holdouts | AUC 0.93679 to 0.94137 |
| High-anxiety subgroup | No positive validation rows in 7 of 8 splits |
| Practical takeaway | Stratify globally, but check subgroup counts separately |

All results below are recomputed from **120,000 sampled training rows**.
Each fit uses 96,000 rows and validates on 24,000. **No GPU, extra datasets,
test predictions or leaderboard submission are needed.**

The notebook includes reusable splitting and fitting functions, two focused
charts and downloadable result tables. ID is excluded from the predictors;
its order is only a stress test, not a timestamp. Repeated holdouts overlap,
so their range is descriptive rather than a confidence interval.

In [ ]:
OUTPUT_NAME = "ev_b_validation"

import os, json, time, hashlib, platform, resource, sys
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

START = time.perf_counter()
override = os.environ.get("EV_DATA_DIR")
roots = [Path(override)] if override else [
    Path("/kaggle/input/competitions/playground-series-s6e9"),
    Path("/kaggle/input/playground-series-s6e9")]
paths = {p.resolve() for p in roots if (p/"train.csv").is_file()}
if len(paths) != 1:
    raise RuntimeError("Attach one official S6E9 competition input, or set EV_DATA_DIR locally")
TRAIN_PATH = next(iter(paths)) / "train.csv"
OUTPUT = Path(OUTPUT_NAME)
OUTPUT.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
def file_hash(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()
manifest = {"observed_at_utc": datetime.now(timezone.utc).isoformat(),
    "train_sha256": file_hash(TRAIN_PATH), "python": platform.python_version(),
    "packages": {p: version(p) for p in ("pandas", "numpy", "scikit-learn", "matplotlib")}}
print("Python", manifest["python"], "|", manifest["packages"])

## 1. Setup and reusable functions

Numeric median imputation/scaling and categorical imputation/one-hot encoding
are fitted only on training rows. Unknown validation levels are handled by
the fitted encoder. The helper explicitly returns NaN for single-class subgroup
AUC: such a score is undefined, not zero and not a failed classifier.

In [ ]:
"""Original train-only experimental helpers for EV-B and EV-C."""
import hashlib
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

TARGET, ID = "Will_Buy_EV", "id"
SAMPLE_SEED = 20260907
SAMPLE_SIZE = 120000
SEEDS = [17, 43, 91]


def load_sample(path, n=SAMPLE_SIZE, seed=SAMPLE_SEED):
    data = pd.read_csv(path)
    if TARGET not in data or ID not in data or not data[ID].is_unique:
        raise ValueError("Expected unique IDs and a training target")
    if data[TARGET].isna().any() or set(data[TARGET].unique()) != {"Yes", "No"}:
        raise ValueError("Expected Yes/No labels")
    if n < 10 or n > len(data):
        raise ValueError("Sample size must be between 10 and dataset size")
    indices = np.arange(len(data))
    if n < len(data):
        indices, _ = train_test_split(indices, train_size=n, random_state=seed, stratify=data[TARGET])
    sample = data.iloc[indices].sort_values(ID).reset_index(drop=True)
    return sample, len(data)


def split_indices(frame, mode, seed=17, fraction=.2):
    idx = np.arange(len(frame))
    if mode in {"stratified", "shuffled"}:
        tr, va = train_test_split(idx, test_size=fraction, random_state=seed,
                                 stratify=frame[TARGET] if mode == "stratified" else None)
    elif mode in {"id_tail", "id_head"}:
        order = np.argsort(frame[ID].to_numpy(), kind="stable")
        count = int(np.ceil(len(frame) * fraction))
        if mode == "id_tail":
            tr, va = order[:-count], order[-count:]
        else:
            tr, va = order[count:], order[:count]
    else:
        raise ValueError("Unknown validation mode")
    assert len(tr) and len(va) and not np.intersect1d(tr, va).size
    assert len(np.union1d(tr, va)) == len(frame)
    return tr, va


def safe_auc(y, p):
    return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan


def pipeline_for(frame, kind):
    numeric = frame.select_dtypes(include="number").columns.tolist()
    categories = [c for c in frame if c not in numeric]
    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if kind == "linear":
        numeric_steps.append(("scale", StandardScaler()))
    prepare = ColumnTransformer([
        ("numeric", Pipeline(numeric_steps), numeric),
        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categories),
    ])
    if kind == "prior":
        model = DummyClassifier(strategy="prior")
    elif kind == "linear":
        model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=500, tol=1e-4)
    elif kind == "tree":
        model = HistGradientBoostingClassifier(max_iter=120, learning_rate=.08,
            max_leaf_nodes=31, min_samples_leaf=40, l2_regularization=1.,
            early_stopping=False, random_state=17)
    else:
        raise ValueError("Unknown model kind")
    return Pipeline([("prepare", prepare), ("model", model)])


def fit_score(sample, tr, va, kind):
    features = [c for c in sample if c not in (TARGET, ID)]
    xtr, xva = sample.iloc[tr][features], sample.iloc[va][features]
    ytr = sample.iloc[tr][TARGET].map({"No": 0, "Yes": 1}).to_numpy()
    yva = sample.iloc[va][TARGET].map({"No": 0, "Yes": 1}).to_numpy()
    if len(np.unique(ytr)) != 2 or len(np.unique(yva)) != 2:
        raise ValueError("Overall training and validation splits must contain both classes")
    model = pipeline_for(xtr, kind)
    with threadpool_limits(limits=2), warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        start = time.perf_counter()
        model.fit(xtr, ytr)
        fit_seconds = time.perf_counter() - start
        start = time.perf_counter()
        predictions = model.predict_proba(xva)[:, list(model.classes_).index(1)]
        predict_seconds = time.perf_counter() - start
    return model, predictions, yva, {
        "train_rows": len(tr), "validation_rows": len(va),
        "train_positive_rate": float(ytr.mean()), "validation_positive_rate": float(yva.mean()),
        "roc_auc": safe_auc(yva, predictions),
        "average_precision": float(average_precision_score(yva, predictions)),
        "log_loss": float(log_loss(yva, predictions, labels=[0, 1])),
        "brier_score": float(brier_score_loss(yva, predictions)),
        "fit_seconds": fit_seconds, "predict_seconds": predict_seconds,
        "convergence_warnings": sum(issubclass(w.category, ConvergenceWarning) for w in caught),
        "other_warnings": sum(not issubclass(w.category, ConvergenceWarning) for w in caught),
    }


def split_fingerprint(sample, tr, va):
    def digest(idx):
        ids = np.sort(sample.iloc[idx][ID].to_numpy()).astype("<i8")
        return hashlib.sha256(ids.tobytes()).hexdigest()
    return {"training_id_sha256": digest(tr), "validation_id_sha256": digest(va)}


def paired_auc_bootstrap(y, candidate, reference, rounds=500, seed=2026):
    y = np.asarray(y)
    positive, negative = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    if not len(positive) or not len(negative) or rounds < 20:
        raise ValueError("Both labels and at least 20 bootstrap rounds are required")
    rng = np.random.default_rng(seed)
    differences = []
    for _ in range(rounds):
        idx = np.concatenate([rng.choice(positive, len(positive), replace=True),
                              rng.choice(negative, len(negative), replace=True)])
        differences.append(roc_auc_score(y[idx], candidate[idx]) - roc_auc_score(y[idx], reference[idx]))
    return np.asarray(differences)

## 2. One sample, eight validation designs

Use sample seed 20260907, split seeds 17/43/91 and a 20% holdout. Three
stratified splits, three ordinary shuffled splits and two ID endpoints share
the same working sample. The JSON report keeps the exact split fingerprints.

In [ ]:
sample, full_rows = load_sample(TRAIN_PATH)
manifest.update({"full_training_rows": full_rows, "sample_rows": len(sample),
    "sampling_seed": SAMPLE_SEED,
    "sample_id_sha256": hashlib.sha256(sample[ID].to_numpy().astype("<i8").tobytes()).hexdigest()})
print(f"Training input: {full_rows:,} rows | Working sample: {len(sample):,} rows | Seed: {SAMPLE_SEED}")
print("Sample label counts:", sample[TARGET].value_counts().to_dict())

In [ ]:
designs = [(mode, seed) for mode in ("stratified", "shuffled") for seed in SEEDS]
designs += [("id_tail", 17), ("id_head", 17)]
partitions = {(mode, seed): split_indices(sample, mode, seed) for mode, seed in designs}
assert np.isnan(safe_auc(np.zeros(3), np.array([.1, .2, .3])))
for (mode, seed), (tr, va) in partitions.items():
    assert len(tr) == 96000 and len(va) == 24000
    assert not np.intersect1d(tr, va).size
print("Eight disjoint train/validation partition checks: PASS")
print("Single-class subgroup AUC control: PASS")

## 3. Compare validation scores

Scores are ordinary holdout scores, not pooled OOF scores. Neither scaling nor
category vocabularies are fitted on the full sample. No early stopping uses
the scored validation data. Runtime includes preprocessing in fit/predict.

In [ ]:
scores, groups, fingerprints = [], [], {}
levels = sorted(sample["Range_Anxiety_Level"].unique())
for mode, seed in designs:
    tr, va = partitions[(mode, seed)]
    model, predictions, yva, result = fit_score(sample, tr, va, "linear")
    key = f"{mode}/{seed}" if mode in ("stratified", "shuffled") else mode
    result.update({"design": key, "mode": mode, "split_seed": seed})
    scores.append(result)
    fingerprints[key] = split_fingerprint(sample, tr, va)
    subgroup = sample.iloc[va]["Range_Anxiety_Level"].to_numpy()
    for level in levels:
        mask = subgroup == level
        yy, pp = yva[mask], predictions[mask]
        groups.append({"design": key, "range_anxiety": level, "rows": int(mask.sum()),
            "positives": int(yy.sum()), "negatives": int(len(yy)-yy.sum()),
            "auc_defined": len(np.unique(yy)) == 2, "roc_auc": safe_auc(yy, pp)})
    print(f"{key}: AUC={result['roc_auc']:.6f}, validation positives={result['validation_positive_rate']:.3%}")
scores = pd.DataFrame(scores)
groups = pd.DataFrame(groups)
assert scores.convergence_warnings.sum() == 0, "Inspect convergence before using scores"
display(scores[["design", "roc_auc", "validation_positive_rate", "fit_seconds", "convergence_warnings"]].round(6))

## 4. A stable class ratio is not a stable score

Stratification stabilizes the overall class ratio; it does not necessarily
minimize AUC variability, guarantee subgroup coverage or represent the unseen
test population. The two horizontal axes are explicitly zoomed.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
colors = scores["mode"].map({"stratified":"#16817A", "shuffled":"#52616B", "id_tail":"#D1603D", "id_head":"#B08D26"})
positions = np.arange(len(scores))
axes[0].scatter(scores.roc_auc, positions, c=colors, s=55)
axes[0].set_yticks(positions, scores.design)
axes[0].set(title="Fixed logistic model", xlabel="Holdout ROC-AUC (zoomed)")
axes[1].scatter(scores.validation_positive_rate*100, positions, c=colors, s=55)
axes[1].set_yticks(positions, scores.design)
axes[1].set(title="Validation label mix", xlabel="Positive rate, percent (zoomed)")
fig.savefig(OUTPUT / "01_split_sensitivity.png", bbox_inches="tight")
plt.show()
grouped = scores.groupby("mode").agg(runs=("roc_auc","size"), auc_min=("roc_auc","min"),
    auc_max=("roc_auc","max"), prevalence_min=("validation_positive_rate","min"),
    prevalence_max=("validation_positive_rate","max"))
display(grouped.round(6))
strat_range = grouped.loc['stratified','auc_max']-grouped.loc['stratified','auc_min']
shuffle_range = grouped.loc['shuffled','auc_max']-grouped.loc['shuffled','auc_min']
display(Markdown(f"In these three seeds, the stratified AUC range is **{strat_range:.6f}** "
    f"and the ordinary-shuffle range is **{shuffle_range:.6f}**. These observed ranges "
    "do not establish that either validation method is generally more stable."))
display(Markdown("These are overlapping repeated holdouts of the same fixed sample. "
    "Do not interpret the min/max spread as a 95% interval or select the highest-scoring split."))

## 5. The rare-group trap

The following is a diagnostic by a synthetic feature, not a fairness or
population-risk claim. Every group is shown, including groups with zero
positives or no rows. AUC requires both labels; a valid numeric AUC based on
very few positives can also be extremely unstable.

In [ ]:
coverage = groups.groupby("range_anxiety").agg(
    validation_runs=("design", "size"),
    undefined_auc=("auc_defined", lambda values: int((~values).sum())),
    fewest_positives=("positives", "min"), most_positives=("positives", "max"))
display(coverage)
matrix = groups.pivot(index="range_anxiety", columns="design", values="auc_defined")
matrix = matrix.reindex(index=levels, columns=scores.design)
positive_counts = groups.pivot(index="range_anxiety", columns="design", values="positives").reindex_like(matrix)
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(12, 3.8), layout="constrained")
ax.imshow(matrix.to_numpy(dtype=int), cmap=ListedColormap(["#E6E7E9", "#16817A"]), vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=30, ha="right")
ax.set_yticks(range(len(matrix.index)), matrix.index)
for i in range(len(matrix.index)):
    for j in range(len(matrix.columns)):
        count = int(positive_counts.iloc[i,j])
        ax.text(j, i, f"{count} positive" + ("" if count==1 else "s"), ha="center", va="center",
                color="white" if matrix.iloc[i,j] else "#202124", fontsize=9)
ax.set_title("Subgroup AUC: teal = both classes; gray = undefined")
fig.savefig(OUTPUT / "02_subgroup_coverage.png", bbox_inches="tight")
plt.show()
display(Markdown(f"Across the listed subgroup/split combinations, **{int((~groups.auc_defined).sum())}** "
    "AUC values are undefined. A teal cell with only one positive is mathematically defined "
    "but should not be treated as a stable subgroup performance estimate."))

## 6. A starting point to reuse

Keep a fixed stratified split for development instead of choosing whichever
split scores highest. The functions above can be reused with another pipeline:

```python
tr, va = split_indices(sample, "stratified", seed=17)
model, probabilities, labels, metrics = fit_score(sample, tr, va, "linear")
```

For rare groups, keep the counts next to the metric and leave single-class
AUC undefined. Use a separate evaluation before claiming a robust model improvement.

In [ ]:
summary = {"candidate":"EV-B", "manifest":manifest, "partition_fingerprints":fingerprints,
    "model":"fixed logistic, C=1, max_iter=500; ID excluded", "split_runs":len(scores),
    "overall_auc_min":float(scores.roc_auc.min()), "overall_auc_max":float(scores.roc_auc.max()),
    "undefined_subgroup_auc_rows":int((~groups.auc_defined).sum()),
    "convergence_warnings":int(scores.convergence_warnings.sum()),
    "test_data_read":False, "submission_created":False, "leaderboard_score":None,
    "total_experiment_seconds":round(time.perf_counter()-START,2),
    "process_peak_rss_mib":round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/(1024**2 if sys.platform=='darwin' else 1024),2)}
scores.to_csv(OUTPUT / "split_scores.csv", index=False)
groups.to_csv(OUTPUT / "subgroup_coverage.csv", index=False)
grouped.to_csv(OUTPUT / "design_summary.csv")
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2, allow_nan=False))
print("Saved: split_scores.csv, subgroup_coverage.csv, design_summary.csv and summary.json")
print(f"Experiment time: {summary['total_experiment_seconds']:.2f}s | No submission created")

## Sources and notes

- [Official S6E9 competition](https://www.kaggle.com/competitions/playground-series-s6e9),
  Yao Yan, Walter Reade and Elizabeth Park, Kaggle, 2026;
  [data](https://www.kaggle.com/competitions/playground-series-s6e9/data) and
  [rules](https://www.kaggle.com/competitions/playground-series-s6e9/rules).
- Earlier [EV-A data-contract audit](https://www.kaggle.com/code/muelsyse111/s6e9-data-audit-and-id-aware-drift)
  found no exact full-feature duplicates. This is not proof that near-duplicates
  or latent groups do not exist.
- Related reading: Georgy Mamarin,
  [S6E9 starter: how to tell a real gain from noise](https://www.kaggle.com/code/georgymamarin/s6e9-starter-how-to-tell-a-real-gain-from-noise).
- Related baseline: evgendvorkin,
  [S6E9 Single XGB CV](https://www.kaggle.com/code/evgendvorkin/s6e9-single-xgb-cv-0-94583).
  These are related reading, not directly comparable score benchmarks. Their code
  is not copied or executed here.
- Models, preprocessing, splitting and AUC/AP/Brier/log-loss metrics use scikit-learn.

Prepared with AI assistance. Only summary tables, plots and reproducibility metadata
are exported. These are training-data experiments, not leaderboard scores or
real-world causal conclusions. Obtain the original data through Kaggle's rules-gated input.